### Configure environment

In [1]:
import chi, os, time, datetime
from chi import lease
from chi import server
from chi import context
from chi import hardware
from chi import network
from chi import ssh

In this section, we configure the Chameleon Python client.

For this experiment, we’re going to use the KVM@TACC site, which we indicate below.

We also need to specify the name of the Chameleon “project” that this experiment is part of. The project name will have the format “CHI-XXXXXX”, where the last part is a 6-digit number, and you can find it on your [user dashboard](https://chameleoncloud.org/user/dashboard/).

In the cell below, select the correct project ID, then run the cell.

In [2]:
context.version = "1.0" 
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv('USER') # all exp resources will have this suffix

# configure openstacksdk for actions unsupported by python-chi
os_conn = chi.clients.connection()


### Configuration for federated learning experiment

In [3]:
username = os.getenv('USER')

node_conf = [
 {'name': "server",   'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-00",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-01",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-02",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-03",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-04",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-05",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-06",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-07",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-08",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}, 
 {'name': "client-09",  'flavor': 'm1.medium', 'image': 'CC-Ubuntu24.04', 'duration': 6, 'packages': []}
]
net_conf = [
     {"name": "net0", "subnet": "10.0.0.0/24", 
      "nodes": [
     {"name": "server",   "addr": "10.0.0.1"}, 
     {"name": "client-00",   "addr": "10.0.0.100"}, 
     {"name": "client-01",   "addr": "10.0.0.101"}, 
     {"name": "client-02",   "addr": "10.0.0.102"}, 
     {"name": "client-03",   "addr": "10.0.0.103"}, 
     {"name": "client-04",   "addr": "10.0.0.104"}, 
     {"name": "client-05",   "addr": "10.0.0.105"}, 
     {"name": "client-06",   "addr": "10.0.0.106"}, 
     {"name": "client-07",   "addr": "10.0.0.107"}, 
     {"name": "client-08",   "addr": "10.0.0.108"}, 
     {"name": "client-09",   "addr": "10.0.0.109"}
     ] }
]

### Configure resources

Now, we will prepare the VMs and network links that our experiment requires.

First, we will prepare a “public” network that we will use for SSH access to our VMs -

In [ ]:
public_net = os_conn.network.create_network(name="public_net_" + username)
public_net_id = public_net.get("id")
public_subnet = os_conn.network.create_subnet(
    name="public_subnet_" + username,
    network_id=public_net.get("id"),
    ip_version='4',
    cidr="192.168.10.0/24",
    gateway_ip="192.168.10.1",
    is_dhcp_enabled = True
)

Next, we will prepare the “experiment” networks -

In [ ]:
nets = []
net_ids = []
subnets = []
for n in net_conf:
    exp_net = os_conn.network.create_network(name="exp_" + n['name']  + '_' + username)
    exp_net_id = exp_net.get("id")
    os_conn.network.update_network(exp_net, is_port_security_enabled=False)
    exp_subnet = os_conn.network.create_subnet(
        name="exp_subnet_" + n['name']  + '_' + username,
        network_id=exp_net.get("id"),
        ip_version='4',
        cidr=n['subnet'],
        gateway_ip=None,
        is_dhcp_enabled = False
    )
    nets.append(exp_net)
    net_ids.append(exp_net_id)
    subnets.append(exp_subnet)

Now we create the VMs -

In [ ]:
servers = []
server_ids = []
for i, n in enumerate(node_conf, start=10):

    # reserve the node
    l = lease.Lease(n['name'] + "_" + username, duration=datetime.timedelta(hours=n.get('duration', 6)))
    l.add_flavor_reservation(id=chi.server.get_flavor_id(n['flavor']), amount=1)
    l.submit(idempotent=True)
    flavor_uuid = l.get_reserved_flavors()[0].id
    
    image_uuid = os_conn.image.find_image(n['image']).id

    # find out details of exp interface(s)
    nics = [{'net-id': chi.network.get_network_id( "exp_" + net['name']  + '_' + username ), 'v4-fixed-ip': node['addr']} for net in net_conf for node in net['nodes'] if node['name']==n['name']]
    # also include a public network interface
    nics.insert(0, {"net-id": public_net_id, "v4-fixed-ip":"192.168.10." + str(i)})
    server = chi.server.create_server(
        server_name=n['name'] + "_" + username,
        image_id=image_uuid,
        flavor_id=flavor_uuid,
        nics=nics
    )
    servers.append(server)
    server_ids.append(chi.server.get_server(n['name'] + "_" + username).id)

We wait for all servers to come up before we proceed -

In [ ]:
for server_id in server_ids:
    chi.server.wait_for_active(server_id)

Next, we will set up SSH access to the VMs.

First, we will make sure the “public” network is connected to the Internet. Then, we will configure it to permit SSH access on port 22 for each port connected to this network.

In [ ]:
# connect them to the Internet on the "public" network (e.g. for software installation)
router = chi.network.create_router('inet_router_' + username, gw_network_name='public')
chi.network.add_subnet_to_router(router.get("id"), public_subnet.get("id"))

In [ ]:
# prepare SSH access on the servers
# WARNING: this relies on undocumented behavior of associate_floating_ip 
# that it associates the IP with the first port on the server
server_ips = []
for server_id in server_ids:
    ip = chi.server.associate_floating_ip(server_id)
    server_ips.append(ip)

Note: The following cells assumes that a security group named “Allow SSH” already exists in your project, and is configured to allow SSH access on port 22. If you have done the “Hello, Chameleon” experiment then you already have this security group.

In [ ]:
security_group_id = os_conn.get_security_group("Allow SSH").id
for port in chi.network.list_ports(): 
    if port['port_security_enabled'] and port['network_id']==public_net.get("id"):
        os_conn.network.update_port(port['id'], security_groups=[security_group_id])

In [ ]:
for ip in server_ips:
    chi.server.wait_for_tcp(ip, port=22)

Finally, we need to configure our resources, including software package installation and network configuration.

In [ ]:
server_remotes = [chi.ssh.Remote(ip) for ip in server_ips]

In [ ]:
# configure an IP address on each port that is supposed to have one
for port in chi.network.list_ports():
    if port['network_id'] in net_ids:
        i = server_ids.index(port['device_id'])
        j = net_ids.index(port['network_id'])
        port_conf =  [item for item in net_conf[j]['nodes'] if item['name'] == node_conf[i]['name'] ][0]
        if port_conf['addr']:
            server_remotes[i].run( "sudo ip addr flush dev $(ip --br link | grep '" + port['mac_address'] + "' | awk '{print $1}')" )
            cmd_prefix = "sudo ip addr add " + port_conf['addr'] + "/" + net_conf[j]['subnet'].split("/")[1] 
            server_remotes[i].run( cmd_prefix + " dev $(ip --br link | grep '" + port['mac_address'] + "' | awk '{print $1}')" )
        else:
            server_remotes[i].run( "sudo ip addr flush dev $(ip --br link | grep '" + port['mac_address'] + "' | awk '{print $1}')" )

In [ ]:
for i, n in enumerate(node_conf):
    remote = server_remotes[i]
    # enable forwarding
    remote.run(f"sudo sysctl -w net.ipv4.ip_forward=1") 
    remote.run(f"sudo firewall-cmd --zone=trusted --add-source=192.168.0.0/16")
    remote.run(f"sudo firewall-cmd --zone=trusted --add-source=172.16.0.0/12")
    remote.run(f"sudo firewall-cmd --zone=trusted --add-source=10.0.0.0/8")
    # configure static routes
    for r in route_conf: 
        if n['name'] in r['nodes']:
            remote.run(f"sudo ip route add " + r['addr'] + " via " + r['gw']) 

In [ ]:
for i, n in enumerate(node_conf):
    # install packages
    if len(n['packages']):
            remote = server_remotes[i]
            remote.run(f"sudo apt update; sudo apt -y install " + " ".join(n['packages'])) 

In [ ]:
# prepare a "hosts" file that has names and addresses of every node
hosts_txt = [ "%s\t%s" % ( n['addr'], n['name'] ) for net in net_conf  for n in net['nodes'] if type(n) is dict and n['addr']]
for remote in server_remotes:
    for h in hosts_txt:
        remote.run("echo %s | sudo tee -a /etc/hosts > /dev/null" % h)

In [ ]:
# turn segment offload off
for port in chi.network.list_ports():
    if port['network_id'] in net_ids:
        i = server_ids.index(port['device_id'])
        j = net_ids.index(port['network_id'])
        for offload in ["gro", "gso", "tso"]:
            server_remotes[i].run( "sudo ethtool -K $(ip --br link | grep '" + port['mac_address'] + "' | awk '{print $1}') " + offload + " off" )

### Draw the network topology

The following cells will draw the network topology, for your reference.

In [ ]:
!pip install networkx

In [ ]:
nodes = [ (n['name'], {'color': 'pink'}) for n in net_conf ] + [(n['name'], {'color': 'lightblue'}) for n in node_conf ]
edges = [(net['name'], node['name'], 
          {'label': node['addr'] + '/' + net['subnet'].split("/")[1] }) if node['addr'] else (net['name'], node['name']) for net in net_conf for node in net['nodes'] ]

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
plt.figure(figsize=(len(nodes),len(nodes)))
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)
pos = nx.spring_layout(G)
nx.draw(G, pos, node_shape='s',  
        node_color=[n[1]['color'] for n in nodes], 
        node_size=[len(n[0])*400 for n in nodes],  
        with_labels=True);
nx.draw_networkx_edge_labels(G,pos,
                             edge_labels=nx.get_edge_attributes(G,'label'),
                             font_color='gray',  font_size=8, rotate=False);

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
slice_info = [{'Name': n['name'], 'SSH command': "ssh cc@" + server_ips[i] } for i, n in enumerate(node_conf)]
pd.DataFrame(slice_info).set_index('Name')

### Install FL code on experiment VMs

In [4]:
server_ids = []
for n in node_conf:
    server_ids.append(chi.server.get_server(n['name'] + "_" + username).id)

server_ips = []
for n in node_conf:
    server_ips.append(chi.server.get_server(n['name'] + "_" + username).get_floating_ip())

server_remotes = [chi.ssh.Remote(ip) for ip in server_ips]

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The py

In [6]:
for remote in server_remotes:
  remote.run("rm -rf federated-chi; git clone https://github.com/Sabrinaaddo07/federated-chi.git")

/opt/conda/lib/python3.12/site-packages/paramiko/client.py:850: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.169: b'c3258112b8212f4b82961e03da493648'
  warnings.warn(
Cloning into 'federated-chi'...
/opt/conda/lib/python3.12/site-packages/paramiko/client.py:850: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.54: b'c2489e27d9f49ae9b37ed9a80b3640be'
  warnings.warn(
Cloning into 'federated-chi'...
/opt/conda/lib/python3.12/site-packages/paramiko/client.py:850: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.153: b'53a5210d13198328eec772677fe738ac'
  warnings.warn(
Cloning into 'federated-chi'...
/opt/conda/lib/python3.12/site-packages/paramiko/client.py:850: UserWarning: Unknown ssh-ed25519 host key for 129.114.27.237: b'c3e4da936e1ec4025763479d698cc076'
  warnings.warn(
Cloning into 'federated-chi'...
/opt/conda/lib/python3.12/site-packages/paramiko/client.py:850: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.84: b'fabaa641681c01379a84d3062eb3e0

In [7]:
for remote in server_remotes:
    remote.run("sudo apt update; sudo apt -y install python3-pip")

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://security.ubuntu.com/ubuntu noble-security/main amd64 c-n-f Metadata [11.5 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/universe amd64 c-n-f Metadata [24.1 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1031 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [1656 kB]
Fetched 2974 kB in 1s (5324 kB/s)
Reading package lists...
Building dependency tree...
Reading state information...
17 packages can be upgraded. Run 'apt list --upgradable' to see them.


Reading package lists...
Building dependency tree...
Reading state information...
python3-pip is already the newest version (24.0+dfsg-1ubuntu1.3).
0 upgraded, 0 newly installed, 0 to remove and 17 not upgraded.


Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists...
Building dependency tree...
Reading state information...
17 packages can be upgraded. Run 'apt list --upgradable' to see them.


Reading package lists...
Building dependency tree...
Reading state information...
python3-pip is already the newest version (24.0+dfsg-1ubuntu1.3).
0 upgraded, 0 newly installed, 0 to remove and 17 not upgraded.


Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists...
Building dependency tree...
Reading state information...
17 packages can be upgraded. Run 'apt list --upgradable' to see them.


...

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  python3-wheel
The following NEW packages will be installed:
  python3-pip python3-wheel
0 upgraded, 2 newly installed, 0 to remove and 17 not upgraded.
Need to get 1373 kB of archives.
After this operation, 7270 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 python3-wheel all 0.42.0-2 [53.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 python3-pip all 24.0+dfsg-1ubuntu1.3 [1320 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 1373 kB in 0s (14.5 MB/s)
Selecting previously unselected package python3-wheel.
(Reading database ... 132149 files and directories currently installed.)
Preparing to unpack .../python3-wheel_0.42.0-2_all.deb ...
Unpacking python3-wheel (0.42.0-2) ...
Selecting previously unselected package python3-pip.
Preparing to unpack .../python3-pip_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip (24.0+dfsg-1ubuntu1.3) ...
Setting up python3-wheel (0.42.0-2) ...
Setting up python3-pip (24.0+dfsg-1ubuntu1.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Restarting services...

Service restarts being deferred:
 /etc/needrestart/restart.d/dbus.service
 systemctl restart networkd-dispatcher.service
 systemctl restart systemd-logind.service
 systemctl restart unattended-upgrades.service

No containers need to be restarted.

User sessions running outdated binaries:
 cc @ session #1: login[1018]
 cc @ session #2: login[1033]
 cc @ user manager service: systemd[1246]

No VM guests are running outdated hypervisor (qemu) binaries on this host.


Pending kernel upgrade
----------------------

Newer kernel available

The currently running kernel version is 6.8.0-111-generic which is not the 
expected kernel version 6.8.0-124-generic.

Restarting the system to load the new kernel will not be handled automatically, 
so you should consider rebooting.



In [8]:
for remote in server_remotes:
  remote.run("cd federated-chi; pip install -r requirements.txt --break-system-packages")

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached flwr-1.31.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached grpcio-1.81.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached grpcio_health_checking-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached cryptography-46.0.7-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.met

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached flwr-1.31.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached grpcio-1.81.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached grpcio_health_checking-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached cryptography-46.0.7-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached flwr-1.31.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached grpcio-1.81.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached grpcio_health_checking-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached cryptography-46.0.7-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 k

...

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached flwr-1.31.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached grpcio-1.81.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached grpcio_health_checking-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached cryptography-46.0.7-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached flwr-1.31.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached grpcio-1.81.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached grpcio_health_checking-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached cryptography-46.0.7-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Run random sampling experiment

In [34]:
for remote in server_remotes:
    try:
        remote.run("sudo killall python3")

python3: no process found


UnexpectedExit: Encountered a bad command exit code!

Command: 'sudo killall python3'

Exit code: 1

Stdout: already printed

Stderr: already printed



In [12]:
server_remotes[0].run(
    "nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u server.py' > server.log 2>&1 &",
    hide=False,
    warn=True,
)

<Result cmd="nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u server.py' > server.log 2>&1 &" exited=0>

In [31]:
server_remotes[0].run("cat server.log")

  RANDOM SAMPLING EXPERIMENT
  10 total clients  |  10 picked per round

  Each round: a random subset of K clients participates.
  Selection is independent every round.

  Open new terminals and run (all 10 before round 1):
    python3 client.py --cid 0 --num_clients 10
    python3 client.py --cid 1 --num_clients 10
    python3 client.py --cid 2 --num_clients 10
    python3 client.py --cid 3 --num_clients 10
    python3 client.py --cid 4 --num_clients 10
    python3 client.py --cid 5 --num_clients 10
    python3 client.py --cid 6 --num_clients 10
    python3 client.py --cid 7 --num_clients 10
    python3 client.py --cid 8 --num_clients 10
    python3 client.py --cid 9 --num_clients 10

  Ctrl+C any client to remove it; server keeps going.
  Ctrl+C the server to stop entirely.

  Server listening on 10.0.0.1:8080
  Waiting for 10 clients...

Port in server address 10.0.0.1:8080 is already in use.


<Result cmd='cat server.log' exited=0>

In [30]:
for cid, remote in enumerate(server_remotes[1:]):
    print(cid)
    remote.run(
        f"nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u client.py --cid {cid} --num_clients 10' > server.log 2>&1 &",
    hide=False,
    warn=True,
    )

0
1
2
3
4
5
6
7
8
9


In [29]:
cid = 0
server_remotes[1].run(
    f"nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u client.py --cid {cid} --num_clients 10' > server.log 2>&1 &",
    hide=False,
    warn=True,
)

<Result cmd="nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u client.py --cid 0 --num_clients 10' > server.log 2>&1 &" exited=0>

In [24]:
server_remotes[1].run(f"cd federated-chi/random_sampling_experiment ; python3 -u client.py --cid {0} --num_clients 10")

--- Client 0 starting ---
  Client 0 loaded 143 training + 36 test samples
  This client's data is PRIVATE — never sent to the server



KeyboardInterrupt: 

In [25]:
server_remotes[1]

<Connection host=129.114.26.54 user=cc>

In [16]:
for cid, remote in enumerate(server_remotes[1:]):
    remote.run(
        f"cd federated-chi/random_sampling_experiment && "
        f"nohup python3 -u client.py --cid {cid} --num_clients 10 "
        f"> client.log 2>&1 &",
        hide=False,
        warn=True,
    )

KeyboardInterrupt: 

### Log in to resources

At this point, we should be able to log in to our resources over SSH! Run the following cell, and observe the output - you will see an SSH command for each of the nodes in your topology.

Now, you can open an SSH session on any of the nodes as follows:

-   In Jupyter, from the menu bar, use File \> New \> Terminal to open a new terminal.
-   Copy an SSH command from the output above, and paste it into the terminal.
-   You can repeat this process (open several terminals) to start a session on each node. Each terminal session will have a tab in the Jupyter environment, so that you can easily switch between them.

Alternatively, you can use your local terminal to log on to each node, if you prefer. (On your local terminal, you may need to also specify your key path as part of the SSH command, using the `-i` argument followed by the path to your private key.)

Now, you can open an SSH session on any of the nodes as follows:

-   In Jupyter, from the menu bar, use File \> New \> Terminal to open a new terminal.
-   Copy an SSH command from the output above, and paste it into the terminal.
-   You can repeat this process (open several terminals) to start a session on each node. Each terminal session will have a tab in the Jupyter environment, so that you can easily switch between them.

Alternatively, you can use your local terminal to log on to each node, if you prefer. (On your local terminal, you may need to also specify your key path as part of the SSH command, using the `-i` argument followed by the path to your private key.)

In [ ]:
%%writefile server.py
import socket

print("Server starting...")

In [ ]:
%%writefile server.py
import flwr as fl

SERVER_ADDRESS = "0.0.0.0:8080"

if __name__ == "__main__":
    fl.server.start_server(
        server_address=SERVER_ADDRESS,
        config=fl.server.ServerConfig(num_rounds=5),
    )

creates server file and "template"; "0.0.0.0:8080" is the way the client can connect to the server and the way the server communicates globally (instead of locally only on my computer)

In [ ]:
import flwr as fl
import argparse

# simple dummy client (you will replace model later)

class FLClient(fl.client.NumPyClient):
    def get_parameters(self, config):
        return []

    def fit(self, parameters, config):
        # simulate local training
        return [], 1, {}

    def evaluate(self, parameters, config):
        return 0.0, 1, {}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--server", type=str, required=True)
    parser.add_argument("--cid", type=int, required=True)
    args = parser.parse_args()

    fl.client.start_numpy_client(
        server_address=args.server,
        client=FLClient()
    )

creates client.py

In [ ]:
server_remotes[0].run("python server.py")

starts the server

In [ ]:
for i in range(1, 11):
    server_remotes[i].run(f"python client.py --server 10.0.0.1:8080 --cid {i-1}")

In [ ]:
starts the clients

### Run all experiments

Run rebuild remotes → auto loop → plot.

In [ ]:
server_ids = []
for n in node_conf:
    server_ids.append(chi.server.get_server(n['name'] + "_" + username).id)

server_ips = []
for n in node_conf:
    server_ips.append(chi.server.get_server(n['name'] + "_" + username).get_floating_ip())

server_remotes = [chi.ssh.Remote(ip) for ip in server_ips]
print("Server remotes rebuilt:", len(server_remotes), "VMs")

In [ ]:
import time

for K in [2, 4, 6, 10]:
    print(f"\n--- K={K} of 10 ---")

    for remote in server_remotes:
        try:
            remote.run("sudo killall python3")
        except:
            pass

    server_remotes[0].run(
        f"nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u server.py --clients_per_round {K}' > server.log 2>&1 &",
        hide=False, warn=True,
    )

    for cid, remote in enumerate(server_remotes[1:]):
        remote.run(
            f"nohup bash -lc 'cd federated-chi/random_sampling_experiment && python3 -u client.py --cid {cid} --num_clients 10' > server.log 2>&1 &",
            hide=False, warn=True,
        )

    for _ in range(200):
        out = server_remotes[0].run(
            "grep -c 'Global test accuracy' server.log 2>/dev/null || echo 0",
            hide=True,
        ).stdout.strip()
        if out:
            print(f"  K={K}: rounds {out}/40", end="\r")
        if out == "40":
            print(f"\n  K={K}: done!")
            break
        time.sleep(10)

print("\nAll experiments finished!")

### Plot results

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt

k_values = [2, 4, 6, 10]
colors = {2: "#f6a623", 4: "#d9534f", 6: "#5cb85c", 10: "#2e86ab"}
markers = {2: "s", 4: "^", 6: "D", 10: "o"}

plt.figure(figsize=(10, 7))

for k in k_values:
    result = server_remotes[0].run(
        f"cat federated-chi/random_sampling_experiment/results_sample_{k}.csv",
        hide=True, warn=True,
    )
    if result.failed or not result.stdout.strip():
        print(f"  Skipping K={k}: file not found")
        continue
    df = pd.read_csv(io.StringIO(result.stdout))
    plt.plot(df["round"], df["global_accuracy"],
             color=colors.get(k, "#888"), marker=markers.get(k, "."),
             linewidth=1.5, markersize=4, label=f"K={k} of 10")
    print(f"  K={k}: final acc = {df['global_accuracy'].iloc[-1]:.4f}")

plt.xlabel("Round", fontsize=13)
plt.ylabel("Global Test Accuracy", fontsize=13)
plt.title("Random Per-Round Client Sampling — Accuracy vs Round",
          fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

### Delete resources

When you finish your experiment, you should delete your resources! The following cells deletes all the resources in this experiment, freeing them for other experimenters.

In [ ]:
# delete the nodes
server_ids = [chi.server.get_server_id(n['name'] + "_" + username) for n in node_conf]
server_ips = [d['addr'] for s in server_ids for d in chi.server.show_server(s).addresses['public_net_' + username] if d['OS-EXT-IPS:type']=='floating']
for server_id in server_ids:
    chi.server.delete_server(server_id)

In [ ]:
# release the floating IP addresses used for SSH
for server_ip in server_ips:
    ip_details = chi.network.get_floating_ip(server_ip)
    chi.neutron().delete_floatingip(ip_details["id"])

In [ ]:
# delete the router used for public Internet access
router = chi.network.get_router("inet_router_" + username)
public_subnet = chi.network.get_subnet("public_subnet_" + username)
public_net = chi.network.get_network("public_net_" + username)
chi.network.remove_subnet_from_router(router.get("id"), public_subnet.get("id"))
chi.network.delete_router(router.get("id"))

In [ ]:
# delete the public network
chi.network.delete_subnet(public_subnet.get('id'))
chi.network.delete_network(public_net.get("id"))

In [ ]:
# delete the experiment networks
subnets = [chi.network.get_subnet("exp_subnet_" + n['name']  + '_' + username) for n in net_conf]
nets    = [chi.network.get_network("exp_" + n['name']  + '_' + username) for n in net_conf]
for subnet, net in zip(subnets, nets):
    chi.network.delete_subnet(subnet.get('id'))
    chi.network.delete_network(net.get('id'))